<a href="https://colab.research.google.com/github/JvResende10/ProcessamentoDigital/blob/main/ProcessamentoDigital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np
from PIL import Image
import base64
import os
from datetime import datetime
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes
import hashlib
from google.colab import files

class SteganographyApp:
    def __init__(self):
        self.private_key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=2048
        )
        self.public_key = self.private_key.public_key()

    def encode_image(self, image_path, message):
        """Embutir texto em uma imagem usando Steganografia"""
        try:
            img = cv2.imread(image_path)

            if img is None:
                raise ValueError(f"Não foi possível ler a imagem: {image_path}")


            full_message = f"START{message}END"

            binary_message = ''.join(format(ord(char), '08b') for char in full_message)

            max_message_length = img.shape[0] * img.shape[1] * 3
            if len(binary_message) > max_message_length:
                raise ValueError(f"Mensagem muito grande. Máximo: {max_message_length} bits")

            encoded_img = img.copy()

            message_index = 0

            for i in range(encoded_img.shape[0]):
                for j in range(encoded_img.shape[1]):
                    for k in range(3):
                        if message_index < len(binary_message):
                            current_bit = int(binary_message[message_index])
                            encoded_img[i, j, k] = (encoded_img[i, j, k] & 0xFE) | current_bit
                            message_index += 1
                        else:
                            break

                    if message_index >= len(binary_message):
                        break

                if message_index >= len(binary_message):
                    break

            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = f'encoded_image_{timestamp}.png'

            cv2.imwrite(output_path, encoded_img)

            print(f"Mensagem codificada: {message}")
            print(f"Comprimento da mensagem: {len(message)}")

            return output_path

        except Exception as e:
            print(f"Erro ao codificar imagem: {e}")
            return None

    def decode_image(self, image_path):
        """Recuperar texto embutido em imagem"""
        try:
            img = cv2.imread(image_path)

            if img is None:
                raise ValueError(f"Não foi possível ler a imagem: {image_path}")

            binary_message = ''

            for i in range(img.shape[0]):
                for j in range(img.shape[1]):
                    for k in range(3):
                        bit = img[i, j, k] & 1
                        binary_message += str(bit)

                        if len(binary_message) >= 8 * 11:
                            try:
                                message_text = ''
                                for x in range(0, len(binary_message), 8):
                                    byte = binary_message[x:x+8]
                                    message_text += chr(int(byte, 2))

                                if message_text.startswith('START') and message_text.endswith('END'):
                                    return message_text[5:-3]
                            except:
                                continue

            return ""

        except Exception as e:
            print(f"Erro ao decodificar imagem: {e}")
            return ""

    def generate_hash(self, image_path):
        """Gerar hash MD5 de uma imagem"""
        try:
            with open(image_path, 'rb') as f:
                image_hash = hashlib.md5(f.read()).hexdigest()
            return image_hash
        except Exception as e:
            print(f"Erro ao gerar hash: {e}")
            return None

    def encrypt_message(self, message):
        """Criptografar mensagem com chave pública"""
        try:
            encrypted = self.public_key.encrypt(
                message.encode(),
                padding.OAEP(
                    mgf=padding.MGF1(algorithm=hashes.SHA256()),
                    algorithm=hashes.SHA256(),
                    label=None
                )
            )
            return base64.b64encode(encrypted).decode()
        except Exception as e:
            print(f"Erro ao criptografar mensagem: {e}")
            return None

    def decrypt_message(self, encrypted_message):
        """Descriptografar mensagem com chave privada"""
        try:
            decoded = base64.b64decode(encrypted_message)
            decrypted = self.private_key.decrypt(
                decoded,
                padding.OAEP(
                    mgf=padding.MGF1(algorithm=hashes.SHA256()),
                    algorithm=hashes.SHA256(),
                    label=None
                )
            )
            return decrypted.decode()
        except Exception as e:
            print(f"Erro ao descriptografar mensagem: {e}")
            return None

    def run(self):
        """Menu principal da aplicação"""
        while True:
            print("\n--- Menu de Steganografia ---")
            print("Arquivos disponíveis:")
            print(os.listdir())

            print("\nOpções:")
            print("(1) Embutir texto em imagem")
            print("(2) Recuperar texto de imagem")
            print("(3) Gerar hash das imagens")
            print("(4) Criptografar mensagem")
            print("(5) Descriptografar mensagem")
            print("(S/s) Sair")

            opcao = input("Escolha uma opção: ")

            if opcao == '1':
                print("Faça o upload da imagem para embutir texto:")
                uploaded = files.upload()

                if uploaded:
                    imagem = list(uploaded.keys())[0]
                    mensagem = input("Digite a mensagem para embutir: ")
                    imagem_codificada = self.encode_image(imagem, mensagem)
                    if imagem_codificada:
                        print(f"Mensagem embutida. Imagem salva em: {imagem_codificada}")
                else:
                    print("Nenhuma imagem foi enviada!")

            elif opcao == '2':
                print("Escolha a fonte da imagem:")
                print("1. Fazer upload")
                print("2. Usar última imagem codificada")
                print("3. Digitar caminho do arquivo")

                escolha = input("Sua escolha: ")

                if escolha == '1':
                    uploaded = files.upload()
                    if uploaded:
                        imagem = list(uploaded.keys())[0]
                    else:
                        print("Nenhuma imagem foi enviada!")
                        continue
                elif escolha == '2':
                    imagens_codificadas = [f for f in os.listdir() if f.startswith('encoded_image') and f.endswith('.png')]
                    if imagens_codificadas:
                        imagem = imagens_codificadas[-1]
                    else:
                        print("Nenhuma imagem codificada encontrada!")
                        continue
                elif escolha == '3':
                    imagem = input("Digite o nome do arquivo: ")

                print(f"Decodificando imagem: {imagem}")

                mensagem = self.decode_image(imagem)

                if mensagem:
                    print(f"Mensagem recuperada: {mensagem}")
                else:
                    print("Não foi possível recuperar a mensagem.")

            elif opcao == '3':
                print("Escolha a fonte da imagem original:")
                print("1. Fazer upload")
                print("2. Digitar caminho do arquivo")

                escolha_original = input("Sua escolha para imagem original: ")

                if escolha_original == '1':
                    uploaded_original = files.upload()
                    if uploaded_original:
                        imagem_original = list(uploaded_original.keys())[0]
                    else:
                        print("Nenhuma imagem original foi enviada!")
                        continue
                elif escolha_original == '2':
                    imagem_original = input("Digite o caminho da imagem original: ")
                    if not os.path.exists(imagem_original):
                        print("Arquivo não encontrado!")
                        continue

                print("\nEscolha a fonte da imagem codificada:")
                print("1. Fazer upload")
                print("2. Digitar caminho do arquivo")

                escolha_codificada = input("Sua escolha para imagem codificada: ")

                if escolha_codificada == '1':
                    uploaded_codificada = files.upload()
                    if uploaded_codificada:
                        imagem_codificada = list(uploaded_codificada.keys())[0]
                    else:
                        print("Nenhuma imagem codificada foi enviada!")
                        continue
                elif escolha_codificada == '2':
                    imagem_codificada = input("Digite o caminho da imagem codificada: ")
                    if not os.path.exists(imagem_codificada):
                        print("Arquivo não encontrado!")
                        continue

                hash_original = self.generate_hash(imagem_original)
                hash_codificada = self.generate_hash(imagem_codificada)

                print(f"Hash original: {hash_original}")
                print(f"Hash codificada: {hash_codificada}")

            elif opcao == '4':
                mensagem = input("Digite a mensagem para criptografar: ")
                mensagem_criptografada = self.encrypt_message(mensagem)
                if mensagem_criptografada:
                    print(f"Mensagem criptografada: {mensagem_criptografada}")

                    print("Faça o upload da imagem para embutir a mensagem criptografada:")
                    uploaded = files.upload()

                    if uploaded:
                        imagem = list(uploaded.keys())[0]
                        imagem_codificada = self.encode_image(imagem, mensagem_criptografada)
                        if imagem_codificada:
                            print(f"Mensagem criptografada embutida. Imagem em: {imagem_codificada}")
                else:
                    print("Erro na criptografia!")

            elif opcao == '5':
                print("Escolha a fonte da imagem com mensagem criptografada:")
                print("1. Fazer upload")
                print("2. Digitar caminho do arquivo")

                escolha = input("Sua escolha: ")

                if escolha == '1':
                    uploaded = files.upload()
                    if uploaded:
                        imagem = list(uploaded.keys())[0]
                    else:
                        print("Nenhuma imagem foi enviada!")
                        continue
                elif escolha == '2':
                    imagem = input("Digite o caminho completo da imagem: ")
                    if not os.path.exists(imagem):
                        print("Arquivo não encontrado!")
                        continue

                try:
                    mensagem_criptografada = self.decode_image(imagem)

                    if mensagem_criptografada:
                        try:
                            mensagem_descriptografada = self.decrypt_message(mensagem_criptografada)
                            if mensagem_descriptografada:
                                print(f"Mensagem descriptografada: {mensagem_descriptografada}")
                            else:
                                print("Não foi possível descriptografar a mensagem.")
                        except Exception as e:
                            print(f"Erro na descriptografia: {e}")
                    else:
                        print("Não foi possível extrair mensagem criptografada da imagem.")

                except Exception as e:
                    print(f"Erro ao processar imagem: {e}")

            elif opcao.upper() == 'S':
                print("Encerrando aplicação...")
                break

            else:
                print("Opção inválida. Tente novamente.")

!pip install opencv-python-headless Pillow cryptography

if __name__ == "__main__":
    app = SteganographyApp()
    app.run()


--- Menu de Steganografia ---
Arquivos disponíveis:
['.config', 'sample_data']

Opções:
(1) Embutir texto em imagem
(2) Recuperar texto de imagem
(3) Gerar hash das imagens
(4) Criptografar mensagem
(5) Descriptografar mensagem
(S/s) Sair
Escolha uma opção: 1
Faça o upload da imagem para embutir texto:


Saving teste.png to teste.png
Digite a mensagem para embutir: Mensagem Secreta
Mensagem codificada: Mensagem Secreta
Comprimento da mensagem: 16
Mensagem embutida. Imagem salva em: encoded_image_20241110_202636.png

--- Menu de Steganografia ---
Arquivos disponíveis:
['.config', 'encoded_image_20241110_202636.png', 'teste.png', 'sample_data']

Opções:
(1) Embutir texto em imagem
(2) Recuperar texto de imagem
(3) Gerar hash das imagens
(4) Criptografar mensagem
(5) Descriptografar mensagem
(S/s) Sair
Escolha uma opção: 2
Escolha a fonte da imagem:
1. Fazer upload
2. Usar última imagem codificada
3. Digitar caminho do arquivo
Sua escolha: 2
Decodificando imagem: encoded_image_20241110_202636.png
Mensagem recuperada: Mensagem Secreta

--- Menu de Steganografia ---
Arquivos disponíveis:
['.config', 'encoded_image_20241110_202636.png', 'teste.png', 'sample_data']

Opções:
(1) Embutir texto em imagem
(2) Recuperar texto de imagem
(3) Gerar hash das imagens
(4) Criptografar mensagem
(5) Descr

Saving teste.png to teste (1).png
Mensagem codificada: gQGBvDIgn+QFvPDglJ0JX41YxDPXcSNq6eZJqdC2+QIojzOrmFnoxWpR4oYj763zKELkSyF0tU3e9i0cTGYAeeiLzfV+6lHeAwbJIkP21Fp8RuqSBWOmzLpfOzv9RuMEj1CxYEYDHC5nePiYQ1wNLBNci7EgfYgROibGqU5rkev9cM158LJSgTs0DVGcwrzc++xdhlAzXIcFMTBiHOwGT1mB349ffewEIjmNqMvqFsr7PusHFvvQS5C5vTmdOsQgRbtXNSmvBAfAfJLc6BjZa5W1jJSUVHHDCWuwTylrY4LG3WpgvkUD/O96Va0nkXKmmO9G+qCv2eMIXZ0mNKWYSQ==
Comprimento da mensagem: 344
Mensagem criptografada embutida. Imagem em: encoded_image_20241110_202943.png

--- Menu de Steganografia ---
Arquivos disponíveis:
['.config', 'encoded_image_20241110_202636.png', 'teste.png', 'teste (1).png', 'encoded_image_20241110_202943.png', 'sample_data']

Opções:
(1) Embutir texto em imagem
(2) Recuperar texto de imagem
(3) Gerar hash das imagens
(4) Criptografar mensagem
(5) Descriptografar mensagem
(S/s) Sair
Escolha uma opção: 5
Escolha a fonte da imagem com mensagem criptografada:
1. Fazer upload
2. Digitar caminho do arquivo
Sua escolha: 2
Digite o cami